# Credit Risk Modeling and Robustness Analysis in Taiwan

In the financial sector, accurately estimating the probability of customer default is critical for balancing profitability, risk exposure, and operational stability.

This project explores credit default prediction using the **Taiwan UCI Credit Card dataset**, which contains `30'000` client records with demographic information, credit limits, billing statements, repayment history, and payment amounts from April to September 2005.

Rather than focusing exclusively on maximizing conventional metrics such as accuracy or ROC-AUC, the primary objective of this work is to develop a **robust, testable, interpretable and economically defensible credit risk model** capable of operating under imperfect and noisy data conditions.

To better reflect real-world financial decision-making, a custom value-based metric was designed. Specifically, the metric assigns a **positive value** to correctly identified non-defaulting customers (True Negatives), and a **loss** proportional to the estimated Exposure at Default (EAD) for undetected defaults (False Negatives). This allows model performance to be evaluated in terms of estimated economic impact and average profit per customer, instead of relying solely on purely statistical indicators.

The project includes:

* Data extraction & Cleaning (*this one*),
* Exploratory data analysis (EDA) & Feature Engineering (`02_EDA_and_engineering.ipynb`)
* Predictive modeling with interpretable and ensemble methods (`03_Preprocessing_and_modeling.ipynb`)
* Robustness evaluation under noisy and partially missing data (`04_Testing_and_deploying.ipynb`)


## 1. Imports and First look

After defining the objective of the project, we begin by importing the required libraries.

In [1]:
import pandas as pd

And then we can load our data with `pandas`

In [2]:
df = pd.read_csv("./data/raw.csv")

df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,2,2,1,37,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,1,2,1,57,-1,0,-1,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ID                          30000 non-null  int64  
 1   LIMIT_BAL                   30000 non-null  float64
 2   SEX                         30000 non-null  int64  
 3   EDUCATION                   30000 non-null  int64  
 4   MARRIAGE                    30000 non-null  int64  
 5   AGE                         30000 non-null  int64  
 6   PAY_0                       30000 non-null  int64  
 7   PAY_2                       30000 non-null  int64  
 8   PAY_3                       30000 non-null  int64  
 9   PAY_4                       30000 non-null  int64  
 10  PAY_5                       30000 non-null  int64  
 11  PAY_6                       30000 non-null  int64  
 12  BILL_AMT1                   30000 non-null  float64
 13  BILL_AMT2                   30000 non-null

Data types are in line with the features and with the original dataset, so we can start cleaning

## 2. Cleaning

The original dataset encodes monthly features using numeric suffixes (e.g., `PAY_0`, `BILL_AMT1`). To improve interpretability, we standardize the naming convention and align all time-based variables to explicit month labels, while also renaming the target column `default.payment.next.month` as `IS_DEFAULT`.

In [4]:
df = df.rename(columns={"PAY_0":"PAY_1"})  # PAY_0 is ambiguous: it appears it should be PAY_1 (to follow the pattern 1-2-...-5-6), so we rename it first. Known issue with the UCI dataset.

cols = ["PAY_1","PAY_2","PAY_3","PAY_4","PAY_5","PAY_6",
        "BILL_AMT1","BILL_AMT2","BILL_AMT3","BILL_AMT4","BILL_AMT5","BILL_AMT6",
        "PAY_AMT1","PAY_AMT2","PAY_AMT3","PAY_AMT4","PAY_AMT5","PAY_AMT6"]

# map number -> month
month_map = {
    1: "SEP",
    2: "AUG",
    3: "JUL",
    4: "JUN",
    5: "MAY",
    6: "APR"
}

def rename_col(col): # rename the column by replacing the number with the month
    for num, month in month_map.items():
        if str(num) in col:
            return col.replace(str(num), month)
    return col

new_cols = [rename_col(c) for c in cols]

col_map = dict(zip(cols, new_cols))

df = df.rename(columns=col_map) # apply the renaming to the dataframe

df["IS_DEFAULT"] = df["default.payment.next.month"]

df = df.drop(columns=["ID", "default.payment.next.month"]) # drop unnecessary columns

df.head()

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_SEP,PAY_AUG,PAY_JUL,PAY_JUN,PAY_MAY,...,BILL_AMTJUN,BILL_AMTMAY,BILL_AMTAPR,PAY_AMTSEP,PAY_AMTAUG,PAY_AMTJUL,PAY_AMTJUN,PAY_AMTMAY,PAY_AMTAPR,IS_DEFAULT
0,20000.0,2,2,1,24,2,2,-1,-1,-2,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,120000.0,2,2,2,26,-1,2,0,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,90000.0,2,2,2,34,0,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,50000.0,2,2,1,37,0,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,50000.0,1,2,1,57,-1,0,-1,0,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0


The dataset reports all monetary variables in New Taiwan Dollars (TWD). For consistency in reporting and as an international standard, all financial features (credit limit, billing amounts, and payments) are converted to USD using a fixed exchange rate of 0.031 USD/TWD.
The rate is an historical extimate for the year of 2005 and hardcoded to ensure reproducibility rather than maximum historical precision.

In [5]:
currency_cols = ["LIMIT_BAL"] + [f"BILL_AMT{month}" for month in ["SEP", "AUG", "JUL", "JUN", "MAY", "APR"]] + [f"PAY_AMT{month}" for month in ["SEP", "AUG", "JUL", "JUN", "MAY", "APR"]]

df[currency_cols] *= 0.031

df.head()

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_SEP,PAY_AUG,PAY_JUL,PAY_JUN,PAY_MAY,...,BILL_AMTJUN,BILL_AMTMAY,BILL_AMTAPR,PAY_AMTSEP,PAY_AMTAUG,PAY_AMTJUL,PAY_AMTJUN,PAY_AMTMAY,PAY_AMTAPR,IS_DEFAULT
0,620.0,2,2,1,24,2,2,-1,-1,-2,...,0.000,0.000,0.000,0.000,21.359,0.0,0.0,0.000,0.000,1
1,3720.0,2,2,2,26,-1,2,0,0,0,...,101.432,107.105,101.091,0.000,31.000,31.0,31.0,0.000,62.000,1
2,2790.0,2,2,2,34,0,0,0,0,0,...,444.261,463.388,482.019,47.058,46.500,31.0,31.0,31.000,155.000,0
3,1550.0,2,2,1,37,0,0,0,0,0,...,877.734,897.729,915.957,62.000,62.589,37.2,34.1,33.139,31.000,0
4,1550.0,1,2,1,57,-1,0,-1,0,0,...,649.140,593.526,593.061,62.000,1137.111,310.0,279.0,21.359,21.049,0


## 3. NaN and Duplicates

After currency conversion, the dataset is checked for missing and duplicated values

In [6]:
print(f"Number of missing values: {df.isna().sum().sum()}")
print(f"Number of duplicate entries: {df.duplicated(keep='first').sum()}")

Number of missing values: 0


Number of duplicate entries: 35


It can be observed that 35 out of 30'000 rows are duplicated. Given their extremely low proportion, they can be removed to prevent potential bias introduced by repeated observations

In [7]:
df = df.drop_duplicates(keep='first')

print(f"Number of new duplicate entries: {df.duplicated(keep='first').sum()}")

Number of new duplicate entries: 0


### IMPORTANT NOTE!!

Under the EU AI Act, credit scoring systems are classified as high-risk (Annex III, point 5(b)), which imposes data governance, bias-testing, and documentation obligations (Articles 9–10).

To reduce compliance burden and the risk of disparate-impact findings, this project takes the conservative approach of excluding pure demographic features (`AGE`, `SEX`, `EDUCATION`, `MARRIAGE`) from the training matrix entirely, rather than retaining them and pursuing formal fairness auditing. This also aligns with the project's modeling philosophy: risk should be assessed from observed repayment behavior, not demographic identity

In [8]:
df = df.drop(columns=["EDUCATION", "MARRIAGE", "SEX", "AGE"])

Finally, the dataset is divided into input features and target variables for subsequent analysis and modeling.

In [9]:
input_cols = ["LIMIT_BAL"] + \
             [f"PAY_{m}" for m in ["SEP","AUG","JUL","JUN","MAY","APR"]] + \
             [f"BILL_AMT{m}" for m in ["SEP","AUG","JUL","JUN","MAY","APR"]] + \
             [f"PAY_AMT{m}" for m in ["SEP","AUG","JUL","JUN","MAY","APR"]]

target_col = "IS_DEFAULT"

## 4. Saving

Data extraction and cleaning are now complete.  
The next notebook, `02_EDA_and_engineering.ipynb`, will focus on exploratory data analysis (EDA) and feature engineering.

In [10]:
df.to_csv("data/cleaned.csv", index=False)